# Accuracy Curve Plot Generation

This notebook generates global test accuracy curves over communication rounds, including standard deviation error bars calculated across 5 experimental seeds.

## 1. Original Plotting Template (Reference)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# 🔧 USER SETTINGS (Original Template)
# ─────────────────────────────────────────────
excel_path = "test.xlsx"
algorithms = ["fedavg", "fedprox", "scaffold", "feddyn", "fedopt", "unicsl"]
seeds = [1, 12, 123, 1234, 42]
datasets = ["A9a", "Run_or_walk_information", "hepmass","magic_gamma_telescope","susy"] 
error_interval = 5

# font settings
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 16,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "legend.fontsize": 16
})

color_map = {
    "fedavg": "#1f77b4",
    "fedprox": "#ff7f0e",
    "scaffold": "#2ca02c",
    "feddyn": "#d62728",
    "fedopt": "#9467bd",
    "unicsl": "#000000",
}

display_names = {
    "fedavg":  "FedAvg",
    "fedprox": "FedProx",
    "scaffold":"SCAFFOLD",
    "feddyn":  "FedDyn",
    "fedopt":  "FedOpt",
    "unicsl":  "UniCSL",
}

# Note: This template reads from a single structured 'test.xlsx'
# df = pd.read_excel(excel_path)
# (Plotting loop matches user's original setup)

## 2. Updated Plotting Code (With pFedMe and Multi-Seed File Reading)

This version parses the 5 separate seed files (`1_experimental_results.xlsx`, etc.), handles the newly added `pfedme` algorithm, and automatically saves each plot as `{dataset}_{scenario}.png`.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# 🔧 SETTINGS FOR NEW EXPERIMENTS
# ─────────────────────────────────────────────
algorithms = ["fedavg", "fedprox", "scaffold", "feddyn", "fedopt", "pfedme", "unicsl_static"]
seeds = [1, 12, 123, 1234, 42]
datasets = ["cifar10", "fashionmnist", "mnist"]
scenarios = {
    "non_iid": "Non-IID",
    "non_iid_corrupted": "Non-IID with noise"
}
error_interval = 5

# font settings
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 16,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "legend.fontsize": 16
})

# Color map with pFedMe included
color_map = {
    "fedavg": "#1f77b4",
    "fedprox": "#ff7f0e",
    "scaffold": "#2ca02c",
    "feddyn": "#d62728",
    "fedopt": "#9467bd",
    "pfedme": "#e377c2",  # Pink/Magenta distinct color
    "unicsl_static": "#000000",
}

display_names = {
    "fedavg":  "FedAvg",
    "fedprox": "FedProx",
    "scaffold":"SCAFFOLD",
    "feddyn":  "FedDyn",
    "fedopt":  "FedOpt",
    "pfedme":  "pFedMe",
    "unicsl_static":  "UniCSL",
}

folder = "."

# ─────────────────────────────────────────────
# 📈 PLOT GENERATION LOOP
# ─────────────────────────────────────────────
for scen, scen_title in scenarios.items():
    # Load all DataFrames for current scenario across seeds
    seed_dfs = {}
    for s in seeds:
        file_name = os.path.join(folder, f"{s}_experimental_results.xlsx")
        if os.path.exists(file_name):
            seed_dfs[s] = pd.read_excel(file_name, sheet_name=scen)
        else:
            print(f"[WARN] File not found: {file_name}")

    for dataset in datasets:
        rounds = np.arange(1, 51)  # communication rounds 1 to 50
        
        algo_mean = {}
        algo_std = {}
        
        for algo in algorithms:
            algo_values = []
            
            for s in seeds:
                if s not in seed_dfs:
                    continue
                df_seed = seed_dfs[s].copy()
                
                # Clean round numbers
                df_seed["Round"] = pd.to_numeric(df_seed["Round"], errors='coerce')
                
                # Filter by dataset
                df_ds = df_seed[df_seed["Dataset"].str.strip().str.lower() == dataset.lower()]
                
                # Extract values for rounds 1 to 50
                seed_vals = []
                for r in rounds:
                    row = df_ds[df_ds["Round"] == r]
                    if not row.empty and algo in row.columns:
                        val = row[algo].values[0]
                        try:
                            seed_vals.append(float(val))
                        except (ValueError, TypeError):
                            seed_vals.append(np.nan)
                    else:
                        seed_vals.append(np.nan)
                algo_values.append(seed_vals)
            
            if not algo_values:
                continue
                
            # Convert to shape (50, num_seeds)
            algo_values = np.array(algo_values).T
            
            # Compute Mean and Standard Deviation across seeds
            algo_mean[algo] = np.nanmean(algo_values, axis=1)
            algo_std[algo] = np.nanstd(algo_values, axis=1, ddof=1) if algo_values.shape[1] > 1 else np.zeros(50)

        # ── Matplotlib Plotting ───────────────────
        plt.figure(figsize=(8, 5))

        for algo in algo_mean:
            mean = algo_mean[algo]
            std = algo_std[algo]
            err_idx = np.arange(0, len(rounds), error_interval)

            plt.plot(
                rounds, mean,
                label=display_names.get(algo, algo.upper()),
                color=color_map.get(algo, None),
                linewidth=3
            )
            plt.errorbar(
                rounds[err_idx], mean[err_idx], yerr=std[err_idx],
                fmt='none', ecolor=color_map.get(algo, None),
                elinewidth=3, capsize=3, alpha=0.8
            )

        plt.xlabel("Communication Round", fontsize=18)
        plt.ylabel("Global Test Accuracy (%)", fontsize=18)
        plt.title(scen_title, fontsize=18)
        plt.grid(True, linestyle="--", alpha=0.5)
        plt.legend(loc='lower right')
        plt.tight_layout()
        
        # Save plot image with dataset_scenario name
        plot_filename = os.path.join(folder, f"{dataset}_{scen}.png")
        plt.savefig(plot_filename, dpi=300)
        plt.close()
        print(f"Successfully saved plot: {plot_filename}")